# 01 - EDA BitcoinHeistData

In [ ]:
# Cell setup chung
from pathlib import Path
from collections import Counter, defaultdict
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_CSV = PROJECT_ROOT / "data" / "raw" / "BitcoinHeistData.csv"
EDA_DIR = PROJECT_ROOT / "outputs" / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)
EDA_PARQUET = EDA_DIR / "bitcoinheist_eda.parquet"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_CSV:", RAW_CSV)
print("EDA_PARQUET:", EDA_PARQUET)

if not RAW_CSV.exists():
    raise FileNotFoundError(
        f"Không thấy file {RAW_CSV}. Hãy đặt BitcoinHeistData.csv vào data/raw/ trước khi chạy notebook."
    )

PROJECT_ROOT: D:\TranQuangSang\HCMUS\Junior\Deep Learning\PROJECT\gnn-active-fraud-detection
RAW_CSV: D:\TranQuangSang\HCMUS\Junior\Deep Learning\PROJECT\gnn-active-fraud-detection\data\raw\BitcoinHeistData.csv
EDA_PARQUET: D:\TranQuangSang\HCMUS\Junior\Deep Learning\PROJECT\gnn-active-fraud-detection\outputs\eda\bitcoinheist_eda.parquet


## 1. Chọn Engine Đọc Dữ Liệu

Ưu tiên `polars` vì có lazy scan và tiết kiệm RAM hơn pandas khi làm EDA trên CSV lớn. Nếu máy chưa có `polars`, notebook tự fallback sang pandas chunks.

Khuyến nghị cài thêm nếu có thể:

```powershell
pip install polars pyarrow
```

In [ ]:
try:
    import polars as pl
    HAS_POLARS = True
except ModuleNotFoundError:
    pl = None
    HAS_POLARS = False

print("HAS_POLARS:", HAS_POLARS)

# Schema dự kiến theo tài liệu SKILL/data_pipeline.md.
# Lưu ý: income có thể rất lớn, ban đầu giữ float64 để tránh mất precision khi EDA range.
PANDAS_DTYPES = {
    "address": "string",
    "year": "int16",
    "day": "int16",
    "length": "float32",
    "weight": "float32",
    "count": "float32",
    "looped": "float32",
    "neighbors": "float32",
    "income": "float64",
    "label": "string",
}

FEATURE_COLS = ["year", "day", "length", "weight", "count", "looped", "neighbors", "income"]
NUMERIC_FEATURES = ["length", "weight", "count", "looped", "neighbors", "income"]
LOG1P_CANDIDATES = ["income", "count", "looped", "length", "weight"]
REQUIRED_COLS = ["address", "year", "day", *NUMERIC_FEATURES, "label"]

## 2. Convert CSV Sang Parquet EDA

CSV lớn đọc đi đọc lại rất chậm. Cell này tạo file parquet tối ưu hơn và thêm 2 cột:

- `label_raw`: nhãn gốc để phân tích ransomware family.
- `label_binary`: `white -> 0`, các ransomware family khác -> `1`.

Cell này **không sửa file raw CSV**.

In [ ]:
def build_eda_parquet_with_polars() -> None:
    schema_overrides = {
        "address": pl.Utf8,
        "year": pl.Int16,
        "day": pl.Int16,
        "length": pl.Float32,
        "weight": pl.Float32,
        "count": pl.Float32,
        "looped": pl.Float32,
        "neighbors": pl.Float32,
        "income": pl.Float64,
        "label": pl.Utf8,
    }
    lf = pl.scan_csv(
        RAW_CSV,
        schema_overrides=schema_overrides,
        infer_schema_length=1000,
    ).with_columns(
        pl.col("label").alias("label_raw"),
        (pl.col("label") != "white").cast(pl.Int8).alias("label_binary"),
    )
    # sink_parquet ghi theo streaming, tránh collect toàn bộ vào RAM nếu Polars hỗ trợ.
    lf.sink_parquet(EDA_PARQUET)


def build_eda_parquet_with_pandas_chunks(chunksize: int = 250_000) -> None:
    # Fallback dùng pandas chunks. Cần pyarrow/fastparquet để ghi parquet.
    # Nếu thiếu pyarrow, hãy cài: pip install pyarrow
    import pyarrow as pa
    import pyarrow.parquet as pq

    writer = None
    try:
        for chunk_idx, chunk in enumerate(pd.read_csv(RAW_CSV, dtype=PANDAS_DTYPES, chunksize=chunksize)):
            chunk["label_raw"] = chunk["label"]
            chunk["label_binary"] = (chunk["label"] != "white").astype("int8")
            table = pa.Table.from_pandas(chunk, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(EDA_PARQUET, table.schema, compression="zstd")
            writer.write_table(table)
            print(f"Đã xử lý chunk {chunk_idx + 1}")
    finally:
        if writer is not None:
            writer.close()


if not EDA_PARQUET.exists():
    if HAS_POLARS:
        build_eda_parquet_with_polars()
    else:
        build_eda_parquet_with_pandas_chunks()
else:
    print("Đã có parquet EDA, bỏ qua bước convert.")

print("Done:", EDA_PARQUET, "size_mb=", round(EDA_PARQUET.stat().st_size / 1024 / 1024, 2))

## 3. Helper Đọc Aggregate

Các cell sau ưu tiên dùng Polars lazy. Nếu không có Polars, dùng pandas đọc các cột cần thiết hoặc chunksize. Tránh đọc full dataframe nếu không cần.

In [ ]:
if HAS_POLARS:
    lf = pl.scan_parquet(EDA_PARQUET)
    print(lf.collect_schema())
else:
    lf = None
    print("Fallback pandas mode. Một số aggregate sẽ dùng chunks để giảm RAM.")


def read_sample(n: int = 100_000, seed: int = 42) -> pd.DataFrame:
    # Dùng sample để vẽ histogram/boxplot, không plot toàn bộ 3 triệu dòng.
    if HAS_POLARS:
        return (
            pl.scan_parquet(EDA_PARQUET)
            .select(["year", "day", *NUMERIC_FEATURES, "label_raw", "label_binary"])
            .collect()
            .sample(n=min(n, 2_916_697), seed=seed, shuffle=True)
            .to_pandas()
        )
    # Với pandas, đọc toàn parquet có thể vẫn ổn sau khi đã tối ưu, nhưng chỉ select columns.
    return pd.read_parquet(
        EDA_PARQUET,
        columns=["year", "day", *NUMERIC_FEATURES, "label_raw", "label_binary"],
    ).sample(n=min(n, 2_916_697), random_state=seed)


## 4. Schema Và Data Quality

Mục tiêu: kiểm tra cột, null, số dòng, duplicate sơ bộ. Duplicate exact rows có thể tốn RAM; với Polars ta tính trực tiếp, với pandas fallback có thể bỏ qua hoặc xử lý chunk/hash sau.

In [ ]:
if HAS_POLARS:
    schema = lf.collect_schema()
    columns = list(schema.names())
    missing_cols = sorted(set([*REQUIRED_COLS, "label_raw", "label_binary"]) - set(columns))
    extra_cols = sorted(set(columns) - set([*REQUIRED_COLS, "label_raw", "label_binary"]))
    row_count = lf.select(pl.len().alias("n_rows")).collect().item()
    null_report = lf.select([pl.col(c).null_count().alias(c) for c in columns]).collect().to_pandas().T
    null_report.columns = ["null_count"]
    duplicate_count = lf.select(pl.struct(columns).is_duplicated().sum().alias("duplicate_rows")).collect().item()
else:
    sample_head = pd.read_parquet(EDA_PARQUET, columns=["address", "year", "day", "label", "label_raw", "label_binary"]).head(5)
    columns = pd.read_parquet(EDA_PARQUET).columns.tolist()
    missing_cols = sorted(set([*REQUIRED_COLS, "label_raw", "label_binary"]) - set(columns))
    extra_cols = sorted(set(columns) - set([*REQUIRED_COLS, "label_raw", "label_binary"]))
    row_count = len(pd.read_parquet(EDA_PARQUET, columns=["label_binary"]))
    null_report = pd.read_parquet(EDA_PARQUET).isna().sum().to_frame("null_count")
    duplicate_count = None

print("n_rows:", row_count)
print("missing_cols:", missing_cols)
print("extra_cols:", extra_cols)
print("duplicate_count:", duplicate_count)
display(null_report)

## 5. Label Distribution

Ta phân tích cả:
- `label_raw`: family gốc.
- `label_binary`: normal/fraud cho bài toán binary classification.

In [ ]:
if HAS_POLARS:
    label_raw_counts = (
        lf.group_by("label_raw")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
        .to_pandas()
    )
    label_binary_counts = (
        lf.group_by("label_binary")
        .agg(pl.len().alias("count"))
        .sort("label_binary")
        .collect()
        .to_pandas()
    )
else:
    label_df = pd.read_parquet(EDA_PARQUET, columns=["label_raw", "label_binary"])
    label_raw_counts = label_df["label_raw"].value_counts().rename_axis("label_raw").reset_index(name="count")
    label_binary_counts = label_df["label_binary"].value_counts().sort_index().rename_axis("label_binary").reset_index(name="count")

label_binary_counts["rate"] = label_binary_counts["count"] / label_binary_counts["count"].sum()
display(label_binary_counts)
display(label_raw_counts.head(30))

fraud_rate = label_binary_counts.loc[label_binary_counts["label_binary"] == 1, "rate"].iloc[0]
print(f"Fraud rate toàn dataset: {fraud_rate:.6f}")

In [ ]:
ax = label_binary_counts.assign(label_name=lambda d: d["label_binary"].map({0: "normal", 1: "fraud"})).plot(
    kind="bar", x="label_name", y="count", legend=False, logy=True, figsize=(6, 4)
)
ax.set_title("Normal vs Fraud count (log scale)")
ax.set_xlabel("label_binary")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

top_families = label_raw_counts[label_raw_counts["label_raw"] != "white"].head(20)
ax = top_families.plot(kind="barh", x="label_raw", y="count", legend=False, figsize=(8, 6))
ax.invert_yaxis()
ax.set_title("Top ransomware families")
ax.set_xlabel("count")
ax.set_ylabel("label_raw")
plt.tight_layout()
plt.show()

## 6. Temporal Analysis

Phần này quyết định split theo thời gian. Cần xem số mẫu, fraud count và fraud rate theo năm/ngày.

In [ ]:
if HAS_POLARS:
    year_summary = (
        lf.group_by("year")
        .agg(
            pl.len().alias("n_rows"),
            pl.col("label_binary").sum().alias("fraud_count"),
            pl.col("label_binary").mean().alias("fraud_rate"),
            pl.col("label_raw").n_unique().alias("n_raw_labels"),
        )
        .sort("year")
        .collect()
        .to_pandas()
    )
    year_day_summary = (
        lf.group_by(["year", "day"])
        .agg(
            pl.len().alias("n_rows"),
            pl.col("label_binary").sum().alias("fraud_count"),
            pl.col("label_binary").mean().alias("fraud_rate"),
        )
        .sort(["year", "day"])
        .collect()
        .to_pandas()
    )
else:
    tmp = pd.read_parquet(EDA_PARQUET, columns=["year", "day", "label_binary", "label_raw"])
    year_summary = tmp.groupby("year").agg(
        n_rows=("label_binary", "size"),
        fraud_count=("label_binary", "sum"),
        fraud_rate=("label_binary", "mean"),
        n_raw_labels=("label_raw", "nunique"),
    ).reset_index()
    year_day_summary = tmp.groupby(["year", "day"]).agg(
        n_rows=("label_binary", "size"),
        fraud_count=("label_binary", "sum"),
        fraud_rate=("label_binary", "mean"),
    ).reset_index()

display(year_summary)
year_summary.to_parquet(EDA_DIR / "year_summary.parquet", index=False)
year_day_summary.to_parquet(EDA_DIR / "year_day_summary.parquet", index=False)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
axes[0].bar(year_summary["year"], year_summary["n_rows"])
axes[0].set_title("Records per year")
axes[0].set_ylabel("n_rows")

axes[1].bar(year_summary["year"], year_summary["fraud_count"], color="tab:red")
axes[1].set_title("Fraud count per year")
axes[1].set_ylabel("fraud_count")

axes[2].plot(year_summary["year"], year_summary["fraud_rate"], marker="o")
axes[2].set_title("Fraud rate per year")
axes[2].set_ylabel("fraud_rate")
axes[2].set_xlabel("year")

plt.tight_layout()
plt.show()

## 7. Feature Distribution

Ta tính summary trên toàn data bằng aggregate, còn plot thì dùng sample để không tốn RAM.

In [ ]:
if HAS_POLARS:
    feature_summary = lf.select(
        [
            pl.col(c).min().alias(f"{c}_min") for c in FEATURE_COLS
        ]
        + [pl.col(c).max().alias(f"{c}_max") for c in FEATURE_COLS]
        + [pl.col(c).mean().alias(f"{c}_mean") for c in FEATURE_COLS]
        + [pl.col(c).std().alias(f"{c}_std") for c in FEATURE_COLS]
        + [pl.col(c).median().alias(f"{c}_median") for c in FEATURE_COLS]
        + [pl.col(c).quantile(0.95).alias(f"{c}_p95") for c in FEATURE_COLS]
        + [pl.col(c).quantile(0.99).alias(f"{c}_p99") for c in FEATURE_COLS]
    ).collect().to_pandas().T
    feature_summary.columns = ["value"]
else:
    tmp_features = pd.read_parquet(EDA_PARQUET, columns=FEATURE_COLS)
    feature_summary = tmp_features.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

display(feature_summary)
feature_summary.to_csv(EDA_DIR / "feature_summary.csv")

In [ ]:
sample_df = read_sample(n=150_000)
print(sample_df.shape)
display(sample_df.head())

In [ ]:
fig, axes = plt.subplots(len(NUMERIC_FEATURES), 2, figsize=(12, 3 * len(NUMERIC_FEATURES)))
for i, col in enumerate(NUMERIC_FEATURES):
    sample_df[col].hist(ax=axes[i, 0], bins=80)
    axes[i, 0].set_title(f"Raw {col}")
    if col in LOG1P_CANDIDATES:
        np.log1p(sample_df[col].clip(lower=0)).hist(ax=axes[i, 1], bins=80, color="tab:green")
        axes[i, 1].set_title(f"log1p({col})")
    else:
        axes[i, 1].axis("off")
plt.tight_layout()
plt.show()

## 8. Feature Vs Label

So sánh normal/fraud trên sample để xem feature nào có tín hiệu phân biệt. Đây chưa phải feature selection cuối cùng.

In [ ]:
grouped_feature_summary = sample_df.groupby("label_binary")[NUMERIC_FEATURES].agg(["mean", "median", "std"])
display(grouped_feature_summary)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.ravel(), NUMERIC_FEATURES):
    plot_values = sample_df[[col, "label_binary"]].copy()
    if col in LOG1P_CANDIDATES:
        plot_values[col] = np.log1p(plot_values[col].clip(lower=0))
    plot_values.boxplot(column=col, by="label_binary", ax=ax, showfliers=False)
    ax.set_title(col + (" log1p" if col in LOG1P_CANDIDATES else ""))
    ax.set_xlabel("label_binary: 0=normal, 1=fraud")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 9. Address-Level Analysis

Mục tiêu: xem address có lặp lại nhiều không, có xuất hiện ở nhiều năm không, và có khả thi để dựng temporal same-address edges không.

Lưu ý RAM: groupby address có thể lớn, nhưng kết quả aggregate theo address vẫn thường nhỏ hơn dữ liệu raw. Nếu máy yếu, hãy chạy cell này sau cùng.

In [ ]:
if HAS_POLARS:
    address_summary = (
        lf.group_by("address")
        .agg(
            pl.len().alias("n_appearances"),
            pl.col("year").min().alias("first_year"),
            pl.col("year").max().alias("last_year"),
            pl.col("label_binary").n_unique().alias("n_binary_labels"),
            pl.col("label_raw").n_unique().alias("n_raw_labels"),
        )
        .collect()
        .to_pandas()
    )
else:
    tmp_addr = pd.read_parquet(EDA_PARQUET, columns=["address", "year", "label_binary", "label_raw"])
    address_summary = tmp_addr.groupby("address").agg(
        n_appearances=("address", "size"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        n_binary_labels=("label_binary", "nunique"),
        n_raw_labels=("label_raw", "nunique"),
    ).reset_index()

display(address_summary["n_appearances"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("unique_addresses:", len(address_summary))
print("repeat_address_rate:", (address_summary["n_appearances"] > 1).mean())
print("multi_year_address_rate:", (address_summary["first_year"] != address_summary["last_year"]).mean())
print("addresses_with_both_binary_labels:", int((address_summary["n_binary_labels"] > 1).sum()))

address_summary.to_parquet(EDA_DIR / "address_summary.parquet", index=False)

## 10. Split Policy Simulation

Thử vài split theo thời gian. Chọn split dựa trên:
- train/val/test đủ fraud để PR-AUC ổn định.
- test là tương lai thật.
- không dùng test cho tuning.
- không split random toàn dataset.

In [ ]:
SPLIT_CANDIDATES = {
    "A_train_2009_2015_val_2016_test_2017_2018": {
        "train": lambda y: y <= 2015,
        "val": lambda y: y == 2016,
        "test": lambda y: y >= 2017,
    },
    "B_train_2013_2015_val_2016_test_2017_2018": {
        "train": lambda y: (y >= 2013) & (y <= 2015),
        "val": lambda y: y == 2016,
        "test": lambda y: y >= 2017,
    },
    "C_train_2009_2016_val_2017_test_2018": {
        "train": lambda y: y <= 2016,
        "val": lambda y: y == 2017,
        "test": lambda y: y >= 2018,
    },
}

# Đọc các cột tối thiểu cho split simulation.
if HAS_POLARS:
    split_base = lf.select(["address", "year", "label_binary", "label_raw"]).collect().to_pandas()
else:
    split_base = pd.read_parquet(EDA_PARQUET, columns=["address", "year", "label_binary", "label_raw"])

rows = []
for split_name, rules in SPLIT_CANDIDATES.items():
    for part, rule in rules.items():
        mask = rule(split_base["year"])
        part_df = split_base.loc[mask]
        n_rows = len(part_df)
        fraud_count = int(part_df["label_binary"].sum()) if n_rows else 0
        rows.append({
            "split_candidate": split_name,
            "part": part,
            "n_rows": n_rows,
            "fraud_count": fraud_count,
            "fraud_rate": fraud_count / max(n_rows, 1),
            "n_raw_labels": part_df["label_raw"].nunique() if n_rows else 0,
            "n_addresses": part_df["address"].nunique() if n_rows else 0,
        })

split_summary = pd.DataFrame(rows)
display(split_summary)
split_summary.to_parquet(EDA_DIR / "split_summary.parquet", index=False)

In [ ]:
# Address overlap giữa các split. Không dùng address làm feature tabular,
# nhưng overlap giúp ta mô tả bài toán transductive/cold-start trong report.
overlap_rows = []
for split_name, rules in SPLIT_CANDIDATES.items():
    sets = {}
    for part, rule in rules.items():
        sets[part] = set(split_base.loc[rule(split_base["year"]), "address"].unique())
    overlap_rows.append({
        "split_candidate": split_name,
        "train_val_overlap": len(sets["train"] & sets["val"]),
        "train_test_overlap": len(sets["train"] & sets["test"]),
        "val_test_overlap": len(sets["val"] & sets["test"]),
    })

overlap_summary = pd.DataFrame(overlap_rows)
display(overlap_summary)
overlap_summary.to_parquet(EDA_DIR / "split_address_overlap.parquet", index=False)

## 11. Graph Feasibility Check

Chưa build graph lớn trong notebook. Chỉ ước lượng nhanh:
- same-address temporal edges có đủ không?
- tỷ lệ node cô lập nếu chỉ dùng same-address edges là bao nhiêu?
- nếu quá sparse, cần kNN/hybrid graph.

In [ ]:
# Ước lượng edge same-address: mỗi address xuất hiện n lần thì tối thiểu có n-1 cạnh theo chuỗi thời gian.
repeat_edges_directed_est = int((address_summary["n_appearances"] - 1).clip(lower=0).sum())
repeat_edges_undirected_est = repeat_edges_directed_est * 2
isolated_rate_same_address = float((address_summary["n_appearances"] == 1).sum() / max(row_count, 1))
avg_degree_est = repeat_edges_undirected_est / max(row_count, 1)

graph_feasibility = pd.DataFrame([
    {
        "strategy": "same_address_temporal_estimate",
        "estimated_edges": repeat_edges_undirected_est,
        "estimated_avg_degree": avg_degree_est,
        "isolated_node_rate_est": isolated_rate_same_address,
        "note": "Nếu avg_degree quá thấp hoặc isolated rate cao, cân nhắc kNN/hybrid graph.",
    }
])
display(graph_feasibility)
graph_feasibility.to_parquet(EDA_DIR / "graph_feasibility.parquet", index=False)

## 12. EDA Summary Template

Điền phần này sau khi chạy xong notebook. Đây là phần quan trọng để quyết định chỉnh code `.py`.

### Kết luận cần chốt

- Dataset shape: `TODO`
- Fraud rate toàn dataset: `TODO`
- Năm có fraud nhiều nhất: `TODO`
- Split được chọn: `TODO`
- Có dùng 2009-2012 không: `TODO`
- Feature cần log1p: `TODO`
- Có clip outlier không: `TODO`
- Có scale `year/day` không: `TODO`
- Baseline đầu tiên nên chạy: `TODO`
- Graph strategy ban đầu: `TODO`

### Quyết định preprocessing dự kiến

```text
label_raw = label
label_binary = 0 nếu label == "white", ngược lại 1
split trước preprocessing train-fit
log1p: income, count, looped, length, weight
scaler.fit: train only
scaler.transform: train/val/test
```

### Việc cần đưa vào code sau EDA

- Cập nhật `bitcoinheist_dataset.py` để dùng `label_binary` thay vì overwrite `label`.
- Tách script prepare data nếu cần convert CSV -> parquet chính thức.
- Cập nhật split policy trong config.
- Cập nhật graph construction nếu same-address temporal quá sparse.
